# Silver: Orders Cleaned
Read bronze orders, apply data quality checks, compute derived columns.

Note: source `total_amount` is already **net of discount** — it is kept as received (exposed as `net_amount`); `gross_amount` and `discount_amount` are derived from it.

In [ ]:
dbutils.widgets.text("catalog", "", "Catalog")
dbutils.widgets.text("schema_bronze", "bronze", "Bronze Schema")
dbutils.widgets.text("schema_silver", "silver", "Silver Schema")
# Plausible order-date range (rows outside it are rejected as data-quality errors)
dbutils.widgets.text("min_order_date", "2024-01-01", "Min Order Date")
dbutils.widgets.text("max_order_date", "2025-12-31", "Max Order Date")

catalog = dbutils.widgets.get("catalog")
schema_bronze = dbutils.widgets.get("schema_bronze")
schema_silver = dbutils.widgets.get("schema_silver")
min_order_date = dbutils.widgets.get("min_order_date")
max_order_date = dbutils.widgets.get("max_order_date")

source_table = f"{catalog}.{schema_bronze}.bronze_orders"
target_table = f"{catalog}.{schema_silver}.silver_orders_cleaned"

In [ ]:
from pyspark.sql.functions import col, current_timestamp, to_date, round as spark_round

df = spark.table(source_table)

df_cleaned = (
    df
    # Quality rules (hard rejects)
    .filter(col("order_id").isNotNull())                   # required business key
    .filter(col("customer_id").isNotNull())                # required foreign key
    .filter(col("quantity") > 0)                           # no zero / negative quantities
    .filter(col("total_amount") >= 0)                      # no negative amounts
    .filter(col("order_datetime").isNotNull())             # every order needs a date
    .filter(to_date(col("order_datetime")).between(min_order_date, max_order_date))  # plausible date range
    # Derived columns — source total_amount is already NET of discount: keep it, never overwrite it
    .withColumn("net_amount", col("total_amount"))
    .withColumn("gross_amount",
        spark_round(col("quantity") * col("unit_price"), 2))
    .withColumn("discount_amount",
        spark_round(col("gross_amount") - col("net_amount"), 2))
    .withColumn("_processed_at", current_timestamp())
)

# overwriteSchema: Silver is a full refresh, so a changed column set must not fail the write
df_cleaned.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table)

row_count = spark.table(target_table).count()
print(f"Cleaned {row_count} rows into {target_table}")

In [ ]:
import json
dbutils.notebook.exit(json.dumps({"status": "SUCCESS", "table": target_table, "rows": row_count}))